In [ ]:
import pandas as pd
df_diamond = pd.read_csv("https://drive.google.com/uc?export=download&id=1m9HU-CoGXCzLtj9DyAoZt-13BfHKQH0c")
df_diamond

In [ ]:
# ============================================#
# ᅷ DIAMOND DYNAMICS - ADVANCED STREAMLIT APP
# ============================================#
%%writefile app.py

import streamlit as st
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.metrics import mean_squared_error, r2_score

from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.tree import DecisionTreeRegressor
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense

# ============================================#
# PAGE CONFIG
# ============================================#
st.set_page_config(page_title="Diamond App", layout="wide")

st.title("💎Diamond Price Prediction & Market Segmentation")

# ============================================#
# LOAD DATA
# ============================================#
@st.cache_data
def load_data():
    df = pd.read_csv("https://drive.google.com/uc?export=download&id=1m9HU-CoGXCzLtj9DyAoZt-13BfHKQH0c")
    return df

df = load_data()

# ============================================#
# DATA CLEANING
# ============================================#
df[['x','y','z']] = df[['x','y','z']].replace(0, np.nan)
df = df.dropna()

# ============================================#
# FEATURE ENGINEERING
# ============================================#
df['price_inr'] = df['price'] * 83
df['volume'] = df['x'] * df['y'] * df['z']
df['price_per_carat'] = df['price'] / df['carat']
df['dimension_ratio'] = (df['x'] + df['y']) / (2 * df['z'])

def carat_cat(x):
    if x < 0.5:
        return 'Light'
    elif x <= 1.5:
        return 'Medium'
    else:
        return 'Heavy'

df['carat_category'] = df['carat'].apply(carat_cat)

# ============================================#
# GLOBAL CATEGORIES FOR ENCODING
# ============================================#
ALL_CUT_CATEGORIES = ['Fair','Good','Very Good','Premium','Ideal']
ALL_COLOR_CATEGORIES = ['D','E','F','G','H','I','J']
ALL_CLARITY_CATEGORIES = ['IF','VVS1','VVS2','VS1','VS2','SI1','SI2','I1']
ALL_CARAT_CATEGORY_LABELS = ['Light', 'Medium', 'Heavy']

# Calculate means for features that are derived from price or need placeholders
MEAN_DEPTH = df['depth'].mean()
MEAN_TABLE = df['table'].mean()
MEAN_PRICE_INR = df['price_inr'].mean()
MEAN_PRICE_PER_CARAT = df['price_per_carat'].mean()

# ============================================#
# SIDEBAR
# ============================================#
st.sidebar.header("ᅴ Navigation")
option = st.sidebar.radio("Select Section",
                          ["EDA", "Model Comparison", "Prediction"])

# ============================================#
# EDA TAB
# ============================================#
if option == "EDA":
    st.header("ᅶ Exploratory Data Analysis")

    col1, col2 = st.columns(2)

    with col1:
        fig, ax = plt.subplots()
        sns.histplot(df['price'], kde=True, ax=ax)
        ax.set_title("Price Distribution")
        st.pyplot(fig)

    with col2:
        fig, ax = plt.subplots()
        sns.histplot(df['carat'], kde=True, ax=ax)
        ax.set_title("Carat Distribution")
        st.pyplot(fig)

    fig, ax = plt.subplots()
    sns.boxplot(x='cut', y='price', data=df, ax=ax)
    st.pyplot(fig)

    fig, ax = plt.subplots(figsize=(8,6))
    sns.heatmap(df.corr(numeric_only=True), annot=True, cmap='coolwarm', ax=ax)
    st.pyplot(fig)

    fig, ax = plt.subplots()
    sns.regplot(x='carat', y='price', data=df, ax=ax)
    st.pyplot(fig)

# ============================================#
# ENCODING
# ============================================#
label_encoders = {}

# Fit each LabelEncoder on the full set of categories it might encounter
le_cut = LabelEncoder()
le_cut.fit(ALL_CUT_CATEGORIES)
df['cut'] = le_cut.transform(df['cut'])
label_encoders['cut'] = le_cut

le_color = LabelEncoder()
le_color.fit(ALL_COLOR_CATEGORIES)
df['color'] = le_color.transform(df['color'])
label_encoders['color'] = le_color

le_clarity = LabelEncoder()
le_clarity.fit(ALL_CLARITY_CATEGORIES)
df['clarity'] = le_clarity.transform(df['clarity'])
label_encoders['clarity'] = le_clarity

le_carat_category = LabelEncoder()
le_carat_category.fit(ALL_CARAT_CATEGORY_LABELS)
df['carat_category'] = le_carat_category.transform(df['carat_category'])
label_encoders['carat_category'] = le_carat_category


# ============================================#
# TRAIN TEST SPLIT
# ============================================#
X = df.drop(['price'], axis=1)
y = df['price']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2)

# ============================================#
# MODELS
# ============================================#
lr = LinearRegression()
rf = RandomForestRegressor()
dt = DecisionTreeRegressor()

lr.fit(X_train, y_train)
rf.fit(X_train, y_train)
dt.fit(X_train, y_train)

# ANN MODEL
ann = Sequential()
ann.add(Dense(64, activation='relu', input_dim=X_train.shape[1]))
ann.add(Dense(32, activation='relu'))
ann.add(Dense(1))

ann.compile(optimizer='adam', loss='mse')
ann.fit(X_train, y_train, epochs=5, verbose=0)

# ============================================#
# MODEL COMPARISON
# ============================================#
if option == "Model Comparison":
    st.header("ᅶ Model Performance")

    models = {
        "Linear Regression": lr.predict(X_test),
        "Decision Tree": dt.predict(X_test),
        "Random Forest": rf.predict(X_test),
        "ANN": ann.predict(X_test).flatten()
    }

    results = []

    for name, pred in models.items():
        r2 = r2_score(y_test, pred)
        rmse = np.sqrt(mean_squared_error(y_test, pred))
        results.append([name, r2, rmse])

    result_df = pd.DataFrame(results, columns=["Model", "R2 Score", "RMSE"])
    st.dataframe(result_df)

    fig, ax = plt.subplots()
    sns.barplot(x="Model", y="R2 Score", data=result_df, ax=ax)
    plt.xticks(rotation=30)
    st.pyplot(fig)

# ============================================#
# CLUSTERING
# ============================================#
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

kmeans = KMeans(n_clusters=3, random_state=42, n_init=10) # Added n_init to suppress warning
df['cluster'] = kmeans.fit_predict(X_scaled)

# PCA
pca = PCA(n_components=2)
pca_data = pca.fit_transform(X_scaled)

# ============================================#
# PREDICTION TAB
# ============================================#
if option == "Prediction":
    st.header("ᅷ Predict Price & Cluster")

    carat = st.number_input("Carat", 0.1, 5.0, 0.5)
    cut_input = st.selectbox("Cut", ALL_CUT_CATEGORIES)
    color_input = st.selectbox("Color", ALL_COLOR_CATEGORIES)
    clarity_input = st.selectbox("Clarity", ALL_CLARITY_CATEGORIES)
    depth_input = st.number_input("Depth", 0.0, 100.0, MEAN_DEPTH)
    table_input = st.number_input("Table", 0.0, 100.0, MEAN_TABLE)
    x = st.number_input("x", 0.0, 10.0, 5.0)
    y_val = st.number_input("y", 0.0, 10.0, 5.0)
    z = st.number_input("z", 0.0, 10.0, 3.0)

    volume = x*y_val*z
    dimension_ratio = (x+y_val)/(2*z) if z!=0 else 0

    if carat < 0.5:
        carat_cat_str = 'Light'
    elif carat <= 1.5:
        carat_cat_str = 'Medium'
    else:
        carat_cat_str = 'Heavy'
    carat_category_encoded = label_encoders['carat_category'].transform([carat_cat_str])[0]

    # Construct input_data matching the exact column order and count of X_train
    # Columns: 'carat', 'cut', 'color', 'clarity', 'depth', 'table', 'x', 'y', 'z',
    #          'price_inr', 'volume', 'price_per_carat', 'dimension_ratio', 'carat_category'
    input_data = np.array([[carat,
                            label_encoders['cut'].transform([cut_input])[0],
                            label_encoders['color'].transform([color_input])[0],
                            label_encoders['clarity'].transform([clarity_input])[0],
                            depth_input,
                            table_input,
                            x, y_val, z,
                            MEAN_PRICE_INR,
                            volume,
                            MEAN_PRICE_PER_CARAT,
                            dimension_ratio,
                            carat_category_encoded]])

    if st.button("Predict"):
        pred = rf.predict(input_data)[0]
        cluster = kmeans.predict(scaler.transform(input_data))[0]

        st.success(f"ᅹ Price: ₹ {round(pred*83,2)}")

        if cluster == 0:
            name = "Affordable Small Diamonds"
        elif cluster == 1:
            name = "Mid-range Balanced Diamonds"
        else:
            name = "Premium Heavy Diamonds"

        st.info(f"ᅶ Cluster: {name}")

    st.subheader("ᅷ Cluster Visualization")

    fig, ax = plt.subplots()
    scatter = ax.scatter(pca_data[:,0], pca_data[:,1], c=df['cluster'])
    st.pyplot(fig)


In [ ]:
!pip install -q streamlit

In [ ]:
!npm install localtunnel

In [ ]:
!streamlit run app.py &>/content/logs.txt & npx localtunnel --port 8501 & curl ipv4.icanhazip.com

34.9.49.150
⠙your url is: https://forty-times-smell.loca.lt


In [ ]:
import pandas as pd
import numpy as np

def load_diamond_data(url):
    """Loads the dataset from a given URL."""
    return pd.read_csv(url)

In [ ]:
def add_inr_price(df, usd_to_inr_rate=83.0):
    """Converts USD price to INR."""
    df['price_inr'] = df['price'] * usd_to_inr_rate
    return df

In [ ]:
def calculate_volume(df):
    """Calculates volume as x * y * z."""
    df['volume'] = df['x'] * df['y'] * df['z']
    return df

In [ ]:
def calculate_price_per_carat(df):
    """Calculates the price per carat."""
    df['price_per_carat'] = df['price'] / df['carat']
    return df

In [ ]:
# Execution Block
if __name__ == "__main__":
    url = "https://drive.google.com/uc?export=download&id=1m9HU-CoGXCzLtj9DyAoZt-13BfHKQH0c"

    # Run the pipeline
    df = load_diamond_data(url)
    df = add_inr_price(df)
    df = calculate_volume(df)
    df = calculate_price_per_carat(df)
    df = calculate_dimension_ratio(df)
    df = categorize_carats(df)
    df = add_bonus_domain_features(df)

    # View the results
    print(df.head())

In [ ]:
def categorize_carats(df):
    """Categorizes diamonds into 'Light', 'Medium', or 'Heavy' based on carat weight."""
    bins = [0, 0.5, 1.5, float('inf')]
    labels = ['Light', 'Medium', 'Heavy']
    df['carat_category'] = pd.cut(df['carat'], bins=bins, labels=labels, right=False)
    return df

In [ ]:
def add_bonus_domain_features(df):
    """Adds bonus domain-specific features like length-to-width ratio and depth-to-table ratio."""
    # Length-to-Width Ratio (x/y)
    df['y_safe'] = df['y'].replace(0, np.nan)
    df['length_width_ratio'] = df['x'] / df['y_safe']
    df.drop(columns=['y_safe'], inplace=True, errors='ignore')

    # Depth-to-Table Ratio
    if 'depth' in df.columns and 'table' in df.columns:
        df['depth_table_ratio'] = df['depth'] / df['table']
    return df

In [ ]:
def calculate_dimension_ratio(df):
    """Calculates the dimension ratio (x + y) / (2 * z), handling division by zero."""
    # Ensure 'z' is not zero to prevent division errors
    df['z_safe'] = df['z'].replace(0, np.nan)
    df['dimension_ratio'] = (df['x'] + df['y']) / (2 * df['z_safe'])
    df.drop(columns=['z_safe'], inplace=True, errors='ignore')
    return df

In [ ]:
# 1. IMPORTS
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Sklearn modules
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OrdinalEncoder
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans
from sklearn.ensemble import RandomForestRegressor
from sklearn.neural_network import MLPRegressor
from sklearn.metrics import mean_squared_error, r2_score

# 2. LOAD DATA
url = "https://drive.google.com/uc?export=download&id=1m9HU-CoGXCzLtj9DyAoZt-13BfHKQH0c"
df = pd.read_csv(url)

# Drop the 'Unnamed: 0' index column if it exists (common in this dataset)
if 'Unnamed: 0' in df.columns:
    df.drop(columns=['Unnamed: 0'], inplace=True)

print("Data Loaded Successfully. Shape:", df.shape)
df

In [ ]:
# 1. FIX ZERO VALUES (Data Cleaning)
# A diamond cannot have 0 length, width, or depth.
df[['x', 'y', 'z']] = df[['x', 'y', 'z']].replace(0, np.nan)
df.dropna(inplace=True) # Drop rows with missing values

# 2. FEATURE ENGINEERING
df['volume'] = df['x'] * df['y'] * df['z']
df['dimension_ratio'] = (df['x'] + df['y']) / (2 * df['z'])
df['length_width_ratio'] = df['x'] / df['y']

# 3. ENCODE CATEGORICAL VARIABLES
# Diamond grades have a specific order (Ordinal), so we shouldn't use random mapping.
cut_order = ['Fair', 'Good', 'Very Good', 'Premium', 'Ideal']
color_order = ['J', 'I', 'H', 'G', 'F', 'E', 'D'] # J is worst, D is best
clarity_order = ['I1', 'SI2', 'SI1', 'VS2', 'VS1', 'VVS2', 'VVS1', 'IF'] # I1 worst, IF best

encoder = OrdinalEncoder(categories=[cut_order, color_order, clarity_order])
df[['cut_encoded', 'color_encoded', 'clarity_encoded']] = encoder.fit_transform(df[['cut', 'color', 'clarity']])

# Drop original text columns to prepare for machine learning
df_ml = df.drop(columns=['cut', 'color', 'clarity'])

print("Feature Engineering & Encoding Complete. Current Shape:", df_ml.shape)

In [ ]:
# 1. SKEWNESS HANDLING (Log Transform on Target)
# Applying log1p (log(1+x)) normalizes the highly right-skewed price distribution
df_ml['price_log'] = np.log1p(df_ml['price'])

# 2. OUTLIER HANDLING (Capping using IQR)
features_to_cap = ['carat', 'depth', 'table', 'volume']

for col in features_to_cap:
    Q1 = df_ml[col].quantile(0.25)
    Q3 = df_ml[col].quantile(0.75)
    IQR = Q3 - Q1
    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR

    # Cap the outliers at the bounds rather than deleting them, preserving data
    df_ml[col] = np.where(df_ml[col] > upper_bound, upper_bound, df_ml[col])
    df_ml[col] = np.where(df_ml[col] < lower_bound, lower_bound, df_ml[col])

print("Outliers capped and Skewness handled.")

In [ ]:
plt.figure(figsize=(12, 8))
# Drop the original 'price' for the heatmap since we are predicting 'price_log'
corr_matrix = df_ml.drop(columns=['price']).corr()

sns.heatmap(corr_matrix, annot=True, cmap='coolwarm', fmt=".2f", linewidths=0.5)
plt.title("Correlation Heatmap of Diamond Features")
plt.show()

# Scatter plot of Carat vs Log Price
plt.figure(figsize=(8, 5))
sns.scatterplot(x='carat', y='price_log', data=df_ml, alpha=0.3, color='blue')
plt.title("Carat vs Log(Price)")
plt.show()

In [ ]:
# 1. SCALING (Crucial for PCA and Clustering)
features_for_clustering = ['carat', 'depth', 'table', 'volume', 'cut_encoded', 'color_encoded', 'clarity_encoded']
scaler = StandardScaler()
X_scaled = scaler.fit_transform(df_ml[features_for_clustering])

# 2. PCA (Dimensionality Reduction to 2 components for visualization)
pca = PCA(n_components=2)
X_pca = pca.fit_transform(X_scaled)
df_ml['PCA1'] = X_pca[:, 0]
df_ml['PCA2'] = X_pca[:, 1]
print(f"Explained Variance by 2 PCA components: {sum(pca.explained_variance_ratio_):.2%}")

# 3. K-MEANS CLUSTERING
# Let's assume 3 market segments
kmeans = KMeans(n_clusters=3, random_state=42, n_init=10)
df_ml['Cluster'] = kmeans.fit_predict(X_scaled)

# Plot the clusters using our PCA components
plt.figure(figsize=(8, 6))
sns.scatterplot(x='PCA1', y='PCA2', hue='Cluster', palette='viridis', data=df_ml, alpha=0.6)
plt.title("Diamond Market Segments (K-Means on PCA Components)")
plt.show()

In [ ]:
# 1. PREPARE TRAIN/TEST DATA
# Drop original price and target price_log. We can include our new 'Cluster' as a feature!
X = df_ml.drop(columns=['price', 'price_log'])
y = df_ml['price_log']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Neural Networks require scaled inputs to converge properly
scaler_ann = StandardScaler()
X_train_scaled = scaler_ann.fit_transform(X_train)
X_test_scaled = scaler_ann.transform(X_test)

# 2. RANDOM FOREST REGRESSOR (Baseline)
rf = RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1)
rf.fit(X_train, y_train)
rf_preds = rf.predict(X_test)

print("--- Random Forest Performance ---")
print(f"R2 Score: {r2_score(y_test, rf_preds):.4f}")
print(f"RMSE (Log Scale): {np.sqrt(mean_squared_error(y_test, rf_preds)):.4f}\n")

# 3. ARTIFICIAL NEURAL NETWORK (MLPRegressor)
# 3 hidden layers with 64, 32, and 16 neurons
ann = MLPRegressor(hidden_layer_sizes=(64, 32, 16), activation='relu', solver='adam',
                   max_iter=500, random_state=42, early_stopping=True)
ann.fit(X_train_scaled, y_train)
ann_preds = ann.predict(X_test_scaled)

print("--- Artificial Neural Network (ANN) Performance ---")
print(f"R2 Score: {r2_score(y_test, ann_preds):.4f}")
print(f"RMSE (Log Scale): {np.sqrt(mean_squared_error(y_test, ann_preds)):.4f}")